In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import glob
from pathlib import Path

import numpy as np
import pandas as pd

import scipy

from aicsimageio import AICSImage, readers

from tqdm import tqdm

from matplotlib import pyplot as plt
import seaborn as sns

import matplotlib.font_manager as font_manager

11-Jun-26 10:28:30 - bfio.backends - WARNING  - Java backend is not available. This could be due to a missing dependency (jpype).


In [2]:
font_path = "/home/z3536241/Fonts/arial.ttf"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name

plt.rcParams['svg.fonttype'] = 'none'

In [3]:
# Convert time points to hours
def convert_timepoint2(time_str):
    days = int(time_str[0:2])
    hours = int(time_str[3:5])
    minutes = int(time_str[6:8])
    total_hours = days * 24 + hours + minutes / 60
    return total_hours

## load quantification for two experiments

In [4]:
quantification_directory_1 = Path("/srv/scratch/berrylab/z3536241/Incucyte/251020_mACmChPOLR2A_Incucyte/QUANTIFICATION")
quantification_directory_2 = Path("/srv/scratch/berrylab/z3536241/Incucyte/251027_mACmChPOLR2A_Incucyte/QUANTIFICATION")

In [5]:
quantification_files_1 = glob.glob(str(quantification_directory_1 / "*.csv"))
quantification_files_2 = glob.glob(str(quantification_directory_2 / "*.csv"))
len(quantification_files_1), len(quantification_files_2)

(12000, 5880)

### assemble dataframe for experiment 1

In [6]:
qdfs = list()


for f in tqdm(quantification_files_1):
    data = pd.read_csv(f)
    data['file'] = Path(f).stem
    qdfs.append(data)

quantifcation_percell_1 = pd.concat(qdfs, ignore_index=True)

files = quantifcation_percell_1['file'].tolist()

times = [x[-9:] for x in files]

wells = [x[8:10] for x in files]

quantifcation_percell_1['times'] = times
quantifcation_percell_1['wells'] = wells

PerCellData_Experiment1 = pd.DataFrame({
    "FILE" : quantifcation_percell_1['file'],
    "WELL" : quantifcation_percell_1['wells'],
    "TIMECODE" : quantifcation_percell_1['times'],
    "CellArea" : quantifcation_percell_1['Channel:0:0_area'],
    "CellBFIntensity" : quantifcation_percell_1['Channel:0:0_intensity_mean_Channel:0:0']
})

100%|██████████| 12000/12000 [03:07<00:00, 64.02it/s]


In [7]:
PerCellData_Experiment1['TIME_HOURS'] = [convert_timepoint2(t) for t in PerCellData_Experiment1.TIMECODE]

row = [w[:1] for w in PerCellData_Experiment1['WELL']]
col = [w[1:] for w in PerCellData_Experiment1['WELL']]

PerCellData_Experiment1['ROW'] = row
PerCellData_Experiment1['COLUMN'] = np.asarray(col).astype(int)

In [8]:
PerWellData_Experiment1 = PerCellData_Experiment1.drop('FILE', axis = 1).groupby(['COLUMN', 'ROW', 'WELL','TIMECODE','TIME_HOURS']).mean().reset_index()

In [9]:
PerWellCounts_Experiment1 = PerCellData_Experiment1.groupby(['COLUMN', 'ROW', 'WELL','TIMECODE','TIME_HOURS']).count().reset_index()
PerWellCounts_Experiment1['TotalNumberCells'] = PerWellCounts_Experiment1['CellArea']
PerWellCounts_Experiment1['LogNumberCells'] = np.log10(PerWellCounts_Experiment1['TotalNumberCells'])
PerWellCounts_Experiment1 = PerWellCounts_Experiment1[['WELL','TIMECODE','TIME_HOURS','TotalNumberCells','LogNumberCells']]
PerWellData_Experiment1 = PerWellData_Experiment1.merge(PerWellCounts_Experiment1)

In [10]:
# map wells to experimental conditions for experiment 1

PerWellData_Experiment1['CELL'] = PerWellData_Experiment1.ROW.map({'B' : 'OsTIRF74G', 
                                           'C' : 'mACPOLR2A', 
                                           'D' : 'mACmChPOLR2A',
                                           'E' : 'OsTIRF74G', 
                                           'F' : 'mACPOLR2A', 
                                           'G' : 'mACmChPOLR2A'})

PerWellData_Experiment1['TREATMENT'] = PerWellData_Experiment1.COLUMN.map({1 : 'Vehicle', 
                                                   2 : '10 µM 5PhIAA', 
                                                   3 : 'Vehicle', 
                                                   4 : '10 µM 5PhIAA', 
                                                   5 : 'Vehicle', 
                                                   6 : '10 µM 5PhIAA', 
                                                   7 : 'Vehicle', 
                                                   8 : '10 µM 5PhIAA', 
                                                   9 : 'Vehicle', 
                                                   10 : '10 µM 5PhIAA', 
                                                   11 : 'Vehicle', 
                                                   12 : '10 µM 5PhIAA', 
                                                  })

PerWellData_Experiment1['SEEDING_DENSITY'] = PerWellData_Experiment1.ROW.map({'B' : 2000, 
                                           'C' : 2000, 
                                           'D' : 2000,
                                           'E' : 3000, 
                                           'F' : 3000, 
                                           'G' : 3000})

### assemble dataframe for experiment 2

In [ ]:
qdfs = list()


for f in tqdm(quantification_files_2):
    data = pd.read_csv(f)
    data['file'] = Path(f).stem
    qdfs.append(data)

quantifcation_percell_2 = pd.concat(qdfs, ignore_index=True)

files = quantifcation_percell_2['file'].tolist()

times = [x[-9:] for x in files]

wells = [x[8:10] for x in files]

quantifcation_percell_2['times'] = times

 48%|████▊     | 2816/5880 [00:40<00:45, 67.67it/s]

In [ ]:
quantifcation_percell_2['wells'] = wells

PerCellData_Experiment2 = pd.DataFrame({
    "FILE" : quantifcation_percell_2['file'],
    "WELL" : quantifcation_percell_2['wells'],
    "TIMECODE" : quantifcation_percell_2['times'],
    "CellArea" : quantifcation_percell_2['Channel:0:0_area'],
    "CellBFIntensity" : quantifcation_percell_2['Channel:0:0_intensity_mean_Channel:0:0']
})

In [ ]:
PerCellData_Experiment2['TIME_HOURS'] = [convert_timepoint2(t) for t in PerCellData_Experiment2.TIMECODE]

row = [w[:1] for w in PerCellData_Experiment2['WELL']]
col = [w[1:] for w in PerCellData_Experiment2['WELL']]

PerCellData_Experiment2['ROW'] = row
PerCellData_Experiment2['COLUMN'] = np.asarray(col).astype(int)

In [ ]:
PerWellData_Experiment2 = PerCellData_Experiment2.drop('FILE', axis = 1).groupby(['COLUMN', 'ROW', 'WELL','TIMECODE','TIME_HOURS']).mean().reset_index()

In [ ]:
PerWellCounts_Experiment2 = PerCellData_Experiment2.groupby(['COLUMN', 'ROW', 'WELL','TIMECODE','TIME_HOURS']).count().reset_index()
PerWellCounts_Experiment2['TotalNumberCells'] = PerWellCounts_Experiment2['CellArea']
PerWellCounts_Experiment2['LogNumberCells'] = np.log10(PerWellCounts_Experiment2['TotalNumberCells'])
PerWellCounts_Experiment2 = PerWellCounts_Experiment2[['WELL','TIMECODE','TIME_HOURS','TotalNumberCells','LogNumberCells']]
PerWellData_Experiment2 = PerWellData_Experiment2.merge(PerWellCounts_Experiment2)

In [ ]:
# map wells to experimental conditions for experiment 1


PerWellData_Experiment2['CELL'] = PerWellData_Experiment2.ROW.map({'B' : 'mACPOLR2A', 
                                           'C' : 'mACmChPOLR2A',
                                           'D' : 'OsTIRF74G', 
                                           'E' : 'mACPOLR2A', 
                                           'F' : 'mACmChPOLR2A',
                                           'G' : 'OsTIRF74G',})

PerWellData_Experiment2['TREATMENT'] = PerWellData_Experiment2.COLUMN.map({1 : '10 µM 5PhIAA', 
                                                   2 : 'Vehicle', 
                                                   3 : '10 µM 5PhIAA', 
                                                   4 : 'Vehicle', 
                                                   5 : '10 µM 5PhIAA', 
                                                   6 : 'Vehicle', 
                                                   7 : '10 µM 5PhIAA', 
                                                   8 : 'Vehicle', 
                                                   9 : '10 µM 5PhIAA', 
                                                   10 : 'Vehicle', 
                                                   11 : '10 µM 5PhIAA', 
                                                   12 : 'Vehicle', 
                                                  })

PerWellData_Experiment2['SEEDING_DENSITY'] = PerWellData_Experiment2.ROW.map({'B' : 2000, 
                                           'C' : 2000, 
                                           'D' : 2000,
                                           'E' : 3000, 
                                           'F' : 3000, 
                                           'G' : 3000})

### combine dataframes across experiments

In [ ]:
## exclude first frame from both experiments
PerWellData_Experiment1 = PerWellData_Experiment1.query('TIME_HOURS > 0')
PerWellData_Experiment2 = PerWellData_Experiment2.query('TIME_HOURS > 0')

In [ ]:
## times off by 1 minute, round to nearest hour and crop last frame of experiment 1 so lengths of experiments are equal

In [ ]:
PerWellData_Experiment1.TIME_HOURS = np.round(PerWellData_Experiment1.TIME_HOURS).astype(int)
PerWellData_Experiment2.TIME_HOURS = np.round(PerWellData_Experiment2.TIME_HOURS).astype(int)

In [ ]:
PerWellData_Experiment1 = PerWellData_Experiment1.query('TIME_HOURS < 98')

In [ ]:
PerWellData_Experiment1['EXPERIMENT'] = '251020'
PerWellData_Experiment2['EXPERIMENT'] = '251027'

In [ ]:
# well B1 in 251027 is excluded as a clear outlier, likely poor drug addition
PerWellData_Experiment2 = PerWellData_Experiment2.query('WELL != "B1"')

In [ ]:
PerWellData = pd.concat([PerWellData_Experiment1, PerWellData_Experiment2])

In [ ]:
# separate by seeding density of 2000 or 3000 cells per well

In [ ]:
PerWellData_2000 = PerWellData.query('SEEDING_DENSITY == 2000')
PerWellData_3000 = PerWellData.query('SEEDING_DENSITY == 3000')

# analyse 2000 cells per well data

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
              col = 'CELL')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'TotalNumberCells',
                hue = 'TREATMENT',
                style = 'EXPERIMENT')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
              col = 'CELL')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT',
                style = 'EXPERIMENT')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
              col = 'CELL')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT',
                style = 'EXPERIMENT',
                errorbar = 'sd')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
              col = 'CELL')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT',
                errorbar = 'sd')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
              col = 'CELL')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'TotalNumberCells',
                hue = 'TREATMENT',
                errorbar = 'sd')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
                  col = 'CELL',
                  row = 'TREATMENT')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT', hue_order = ['Vehicle', '10 µM 5PhIAA'],
                marker = 'o',
                errorbar = 'sd')

g.add_legend()

g.set_titles('{col_name}')

for i in range(3):
    for j in range(2):
        g.axes[j][i].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

plt.show()

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
                  col = 'CELL')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT', hue_order = ['Vehicle', '10 µM 5PhIAA'],
                marker = 'o',
                errorbar = 'sd')

g.add_legend()

g.set_titles('{col_name}')

for i in range(3):
        g.axes[0][i].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

plt.show()

In [ ]:
sns.lineplot(PerWellData_2000.query('TREATMENT == "Vehicle"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'CELL',
             errorbar = 'sd',
             marker = 'o')

## calculate growth rates in vehicle treated cells

In [ ]:
PerWellData_2000_Vehicle = PerWellData_2000.query('TREATMENT == "Vehicle"')

f, a = plt.subplots(ncols = 3, figsize = (12,4))

ls = []

for e_i, e in enumerate(PerWellData_2000_Vehicle.EXPERIMENT.unique()):

    e_frame = PerWellData_2000_Vehicle[PerWellData_2000_Vehicle.EXPERIMENT == e]

    for c_i, c in enumerate(e_frame.CELL.unique()):
    
        c_frame = e_frame[e_frame.CELL == c]
    
        for w in c_frame.WELL.unique():
    
            w_frame = c_frame[c_frame.WELL == w]
    
            x_vals = w_frame.TIME_HOURS.tolist()
            y_vals = w_frame.LogNumberCells.tolist()
    
            l = scipy.stats.linregress(x = x_vals, y = y_vals)
    
            sl = l.slope
            intr = l.intercept
    
            keys = ["EXPERIMENT", "WELL", "CELL", "SLOPE", "INTERCEPT", "r", "p", "STD", "INTERCEPT_STD"]
            vals = [[e], [w], [c], [l.slope], [l.intercept], [l.rvalue], [l.pvalue], [l.stderr], [l.intercept_stderr]]
    
            f = pd.DataFrame( dict(zip(keys,vals)) )
            ls.append(f)
    
            sns.scatterplot(x = x_vals, y = y_vals, ax = a[c_i], size = 4, legend = False)
            a[c_i].axline( (0,intr), slope = sl, color = 'black')
    
            #a[c_i].set_xlim(0,6.5)
            #a[c_i].set_ylim(7.5,9.5)
    
            a[c_i].set_title(str(c))

VehicleTreated_GrowthRates_2000 = pd.concat(ls)

In [ ]:
cell_order = ['OsTIRF74G', 'mACmChPOLR2A', 'mACPOLR2A']

h = sns.swarmplot(VehicleTreated_GrowthRates_2000,
              x = 'CELL', order = cell_order,
              y = 'SLOPE',
              hue = 'CELL',
              dodge = False)

sns.barplot(VehicleTreated_GrowthRates_2000,
            x = 'CELL', order = cell_order,
            y = 'SLOPE',
            hue = 'CELL',
            dodge = False,
            errorbar = None,
            alpha = 0.4)

sns.move_legend(h, 'upper left', bbox_to_anchor = (1,1))

plt.ylabel('Growth rate /hr')

## calculate growth rates after vehicle or 5PhIAA addition

In [ ]:
PerWellData_2000_AfterTreatment = PerWellData_2000.query('TIME_HOURS >= 48')

f, a = plt.subplots(ncols = 3, nrows = 2, figsize = (6,6))

ls = []

for e_i, e in enumerate(PerWellData_2000_AfterTreatment.EXPERIMENT.unique()):

    e_frame = PerWellData_2000_AfterTreatment[PerWellData_2000_AfterTreatment.EXPERIMENT == e]

    for c_i, c in enumerate(e_frame.CELL.unique()):
    
        c_frame = e_frame[e_frame.CELL == c]

        for t_i, t in enumerate(c_frame.TREATMENT.unique()):
    
            t_frame = c_frame[c_frame.TREATMENT == t]
    
            for w in t_frame.WELL.unique():
        
                w_frame = t_frame[t_frame.WELL == w]
        
                x_vals = w_frame.TIME_HOURS.tolist()
                y_vals = w_frame.LogNumberCells.tolist()
        
                l = scipy.stats.linregress(x = x_vals, y = y_vals)
        
                sl = l.slope
                intr = l.intercept
        
                keys = ["EXPERIMENT", "WELL", "CELL", "TREATMENT", "SLOPE", "INTERCEPT", "r", "p", "STD", "INTERCEPT_STD"]
                vals = [[e], [w], [c], [t], [l.slope], [l.intercept], [l.rvalue], [l.pvalue], [l.stderr], [l.intercept_stderr]]
        
                f = pd.DataFrame( dict(zip(keys,vals)) )
                ls.append(f)
        
                sns.scatterplot(x = x_vals, y = y_vals, ax = a[e_i,c_i], size = 4, legend = False)
                a[e_i,c_i].axline( (0,intr), slope = sl, color = 'black')
        
                a[e_i,c_i].set_ylim(1,4)
        
                a[e_i,c_i].set_title(str(c))

AfterTreatment_GrowthRates_2000 = pd.concat(ls)

In [ ]:
sns.swarmplot(AfterTreatment_GrowthRates_2000,
              x = 'CELL', order = cell_order,
              y = 'SLOPE',
              hue = 'TREATMENT',
              dodge = True)

h = sns.barplot(AfterTreatment_GrowthRates_2000,
              x = 'CELL', order = cell_order,
              y = 'SLOPE',
              hue = 'TREATMENT',
              dodge = True,
            errorbar = None,
            alpha = 0.4)

sns.move_legend(h, 'upper left', bbox_to_anchor = (1,1))

plt.ylim(-0.005,0.025)

plt.ylabel('Growth rate /hr')

In [ ]:
sns.set_context('paper')

pal = sns.color_palette(['grey', 'xkcd:sky blue'])

f, a = plt.subplots(ncols = 3, figsize = (6.5,2))

sns.lineplot(PerWellData_2000.query('CELL == "OsTIRF74G"'),
             x = 'TIME_HOURS',
             y = 'TotalNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[0],
             legend = False,
             linewidth = 2)

sns.lineplot(PerWellData_2000.query('CELL == "mACPOLR2A"'),
             x = 'TIME_HOURS',
             y = 'TotalNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[1],
             legend = False,
             linewidth = 2)

h = sns.lineplot(PerWellData_2000.query('CELL == "mACmChPOLR2A"'),
             x = 'TIME_HOURS',
             y = 'TotalNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[2],
             linewidth = 2)

sns.move_legend(h, 'upper left', bbox_to_anchor = (1,1))

a[0].set_ylabel('Number of cells')
a[1].set_ylabel('')
a[2].set_ylabel('')

a[0].set_title('OsTIRF74G')
a[1].set_title('mAC-POLR2A')
a[2].set_title('mAC-/mCh-POLR2A')

for i in range(3):
    a[i].set_box_aspect(1)
    a[i].axvline(48, color = 'xkcd:dark grey', alpha = 0.5, linestyle = '--')
    a[i].set_ylim(0,4000)
    a[i].set_yticks(np.linspace(0,4000,3))
    a[i].set_xlim(0,98)
    a[i].set_xticks(np.linspace(0,96,5))
    a[i].set_xlabel('Time (hours)')

plt.tight_layout()

#plt.savefig('mAC_mACmCh_Incucyte_numbercells.svg')

In [ ]:
sns.set_context('paper')

pal = sns.color_palette(['grey', 'xkcd:dark sky blue'])

f, a = plt.subplots(ncols = 3, figsize = (8,1.5), layout = 'constrained')

sns.lineplot(PerWellData_2000.query('CELL == "OsTIRF74G"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[0],
             legend = False,
             linewidth = 2)


sns.lineplot(PerWellData_2000.query('CELL == "mACPOLR2A"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[1],
             legend = False,
             linewidth = 2)

h = sns.lineplot(PerWellData_2000.query('CELL == "mACmChPOLR2A"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[2],
             linewidth = 2)

sns.move_legend(h, 'upper left', bbox_to_anchor = (1,1))

a[1].set_ylabel('$\mathdefault{log_{10}}$ (#cells)', fontsize = 8)
a[0].set_ylabel('')
a[2].set_ylabel('')


for i in range(3):
    a[i].set_box_aspect(1)
    a[i].axvline(48, color = 'xkcd:dark grey', alpha = 0.5, linestyle = '--')
    a[i].set_ylim(1,4)
    a[i].set_xlim(0,98)
    a[i].set_xticks(np.linspace(0,96,5))
    a[i].set_xlabel('Time (hours)', fontsize = 8)
    a[i].tick_params(axis='both', labelsize=8)

plt.tight_layout()

#plt.savefig('mAC_mACmCh_Incucyte_numbercells_log.svg')

In [ ]:
# subtract 48 hours to make drug addition point == 0 hours

PerWellData_2000['TIME_HOURS_SUBTRACTED'] = PerWellData_2000['TIME_HOURS'] - 48

In [ ]:
sns.set_context('paper')

pal = sns.color_palette(['grey', 'xkcd:dark sky blue'])

f, a = plt.subplots(ncols = 3, figsize = (8,1.5))

sns.lineplot(PerWellData_2000.query('CELL == "OsTIRF74G"'),
             x = 'TIME_HOURS_SUBTRACTED',
             y = 'LogNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[0],
             legend = False,
             linewidth = 2)


sns.lineplot(PerWellData_2000.query('CELL == "mACPOLR2A"'),
             x = 'TIME_HOURS_SUBTRACTED',
             y = 'LogNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[1],
             legend = False,
             linewidth = 2)

h = sns.lineplot(PerWellData_2000.query('CELL == "mACmChPOLR2A"'),
             x = 'TIME_HOURS_SUBTRACTED',
             y = 'LogNumberCells',
             hue = 'TREATMENT', palette = pal,
             errorbar = 'sd',
             ax = a[2],
             linewidth = 2)

sns.move_legend(h, 'upper left', bbox_to_anchor = (1,1))

a[1].set_ylabel('$\mathdefault{log_{10}}$ (#cells)', fontsize = 8)
a[0].set_ylabel('')
a[2].set_ylabel('')


for i in range(3):
    a[i].set_box_aspect(1)
    a[i].axvline(0, color = 'xkcd:dark grey', alpha = 0.5, linestyle = '--')
    a[i].set_ylim(1,4)
    a[i].set_xlim(-48,48)
    a[i].set_xticks(np.linspace(-48,48,5))
    a[i].set_xlabel('Time (hours)', fontsize = 8)
    a[i].tick_params(axis='both', labelsize=8)

plt.tight_layout()

plt.savefig('mAC_mACmCh_Incucyte_numbercells_log_timesub.svg')

In [ ]:
sns.set_context('paper')

f, a = plt.subplots(ncols = 3)

sns.lineplot(PerWellData_2000.query('CELL == "OsTIRF74G"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'TREATMENT',
             errorbar = 'sd',
             ax = a[0],
             legend = False)

sns.lineplot(PerWellData_2000.query('CELL == "mACPOLR2A"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'TREATMENT',
             errorbar = 'sd',
             ax = a[1],
             legend = False)

sns.lineplot(PerWellData_2000.query('CELL == "mACmChPOLR2A"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'TREATMENT',
             errorbar = 'sd',
             ax = a[2],
             legend = False)


for i in range(3):
    a[i].set_box_aspect(1)
    a[i].axvline(48, color = 'xkcd:dark grey', alpha = 0.5, linestyle = '--')
    a[i].set_ylim(1,4)
    a[i].set_xlim(0,98)
    a[i].set_xticks(np.linspace(0,96,5))

plt.tight_layout()

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
              col = 'CELL')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'TotalNumberCells',
                hue = 'TREATMENT',
                errorbar = 'sd')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_3000,
              col = 'CELL')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'WELL')

In [ ]:
g = sns.FacetGrid(PerWellData_3000,
              col = 'CELL')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT')

for j in range(3):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

In [ ]:
g = sns.FacetGrid(PerWellData_3000.query('CELL == "mACPOLR2A"'),
                  col = 'TREATMENT')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'WELL')


for j in range(2):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

plt.show()

In [ ]:
g = sns.FacetGrid(PerWellData_3000,
              col = 'CELL',
                  row = 'TREATMENT')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'TotalNumberCells',
                hue = 'WELL')

for i in range(3):
    for j in range(2):
        g.axes[j][i].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

plt.show()

In [ ]:
g = sns.FacetGrid(PerWellData_3000.query('CELL == "mACPOLR2A"'),
                  col = 'TREATMENT')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'TotalNumberCells',
                hue = 'WELL')


for j in range(2):
    g.axes[0][j].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

plt.show()

In [ ]:
g = sns.FacetGrid(PerWellData_3000,
              col = 'CELL',
                  row = 'TREATMENT')

g.map_dataframe(sns.scatterplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'WELL')

In [ ]:
g = sns.FacetGrid(PerWellData_2000,
                  col = 'CELL',
                  row = 'TREATMENT')

g.map_dataframe(sns.lineplot,
                x = 'TIME_HOURS',
                y = 'LogNumberCells',
                hue = 'TREATMENT', hue_order = ['Vehicle', '10 µM 5PhIAA'],
                marker = 'o',
                errorbar = 'sd')

g.add_legend()

g.set_titles('{col_name}')

for i in range(3):
    for j in range(2):
        g.axes[j][i].axvline(48, color = 'xkcd:dark grey', linestyle = '--')

plt.show()

In [ ]:
sns.lineplot(PerWellData_2000.query('TREATMENT == "Vehicle"'),
             x = 'TIME_HOURS',
             y = 'LogNumberCells',
             hue = 'CELL',
             errorbar = 'sd',
             marker = 'o')

## calculate growth rates in vehicle treated cells

In [ ]:
PerWellData_2000_Vehicle = PerWellData_2000.query('TREATMENT == "Vehicle"')

f, a = plt.subplots(ncols = 3, figsize = (12,4))

ls = []

for c_i, c in enumerate(PerWellData_2000_Vehicle.CELL.unique()):

    c_frame = PerWellData_2000_Vehicle[PerWellData_2000_Vehicle.CELL == c]

    for w in c_frame.WELL.unique():

        w_frame = c_frame[c_frame.WELL == w]

        x_vals = w_frame.TIME_HOURS.tolist()
        y_vals = w_frame.LogNumberCells.tolist()

        l = scipy.stats.linregress(x = x_vals, y = y_vals)

        sl = l.slope
        intr = l.intercept

        keys = ["Well", "Cell", "slope", "intercept", "rvalue", "pvalue", "stderr", "intercept_stderr"]
        vals = [[w], [c], [l.slope], [l.intercept], [l.rvalue], [l.pvalue], [l.stderr], [l.intercept_stderr]]

        f = pd.DataFrame( dict(zip(keys,vals)) )
        ls.append(f)

        sns.scatterplot(x = x_vals, y = y_vals, ax = a[c_i], size = 4, legend = False)
        a[c_i].axline( (0,intr), slope = sl, color = 'black')

        #a[c_i].set_xlim(0,6.5)
        #a[c_i].set_ylim(7.5,9.5)

        a[c_i].set_title(str(c))

VehicleTreated_GrowthRates_2000 = pd.concat(ls)

In [ ]:
sns.swarmplot(VehicleTreated_GrowthRates_2000,
              x = 'Cell',
              y = 'slope',
              hue = 'Cell',
              dodge = False)

sns.barplot(VehicleTreated_GrowthRates_2000,
              x = 'Cell',
              y = 'slope',
              hue = 'Cell',
              dodge = False,
            errorbar = None,
            alpha = 0.4)

plt.ylim(-0.005,0.025)

plt.ylabel('Growth rate /hr')

## calculate growth rates after vehicle or 5PhIAA addition

In [ ]:
PerWellData_2000_AfterTreatment = PerWellData_2000.query('TIME_HOURS >= 48')

f, a = plt.subplots(ncols = 3, figsize = (12,4))

ls = []

for c_i, c in enumerate(PerWellData_2000_AfterTreatment.CELL.unique()):

    c_frame = PerWellData_2000_AfterTreatment[PerWellData_2000_AfterTreatment.CELL == c]

    for t_i, t in enumerate(c_frame.TREATMENT.unique()):

        t_frame = c_frame[c_frame.TREATMENT == t]

        for w in t_frame.WELL.unique():
    
            w_frame = t_frame[t_frame.WELL == w]
    
            x_vals = w_frame.TIME_HOURS.tolist()
            y_vals = w_frame.LogNumberCells.tolist()
    
            l = scipy.stats.linregress(x = x_vals, y = y_vals)
    
            sl = l.slope
            intr = l.intercept
    
            keys = ["Well", "Cell", "Treatment", "slope", "intercept", "rvalue", "pvalue", "stderr", "intercept_stderr"]
            vals = [[w], [c], [t], [l.slope], [l.intercept], [l.rvalue], [l.pvalue], [l.stderr], [l.intercept_stderr]]
    
            f = pd.DataFrame( dict(zip(keys,vals)) )
            ls.append(f)
    
            sns.scatterplot(x = x_vals, y = y_vals, ax = a[c_i], size = 4, legend = False)
            a[c_i].axline( (0,intr), slope = sl, color = 'black')
    
            #a[c_i].set_xlim(0,6.5)
            #a[c_i].set_ylim(7.5,9.5)
    
            a[c_i].set_title(str(c))

AfterTreatment_GrowthRates_2000 = pd.concat(ls)

In [ ]:
sns.set_context('paper')

f, a = plt.subplots(figsize = (3,2), layout = 'constrained')

sns.swarmplot(AfterTreatment_GrowthRates_2000,
              x = 'Cell',
              y = 'slope',
              hue = 'Treatment',  palette = pal,
              dodge = True,
              legend = False,
              s = 4)

h = sns.barplot(AfterTreatment_GrowthRates_2000,
              x = 'Cell',
              y = 'slope',
              hue = 'Treatment',  palette = pal,
              dodge = True,
            errorbar = None,
            alpha = 0.4)

h.legend().remove()

plt.ylim(-0.005,0.025)
plt.yticks(np.linspace(0,0.02,3))

plt.xticks(rotation = 90)

a.set_box_aspect(0.7)

plt.ylabel('Growth rate ($\mathregular{hr^{-1}}$)', fontsize = 8)

plt.tick_params(axis = 'both', labelsize = 8)

plt.savefig('mAC_mACmCh_Incucyte_growthrates.svg')